# Прототип раннього виявлення навчального ризику
Практична робота №6. Інтелектуальна система онлайн-освіти. Кожна клітинка працює в Python 3 зі стандартною бібліотекою. Запустіть усі клітинки послідовно. Приклади синтетичні.

## Компонент C6 і правила розгляду C7
Індекс є зваженим балом, а 45 і 60 визначають маршрути. Звернення щодо підтримки потребує рішення викладача.

In [1]:
"""Explainable early warning component C6 and demonstration routing for C7."""

from decimal import Decimal, InvalidOperation, ROUND_HALF_UP

VERSION = 'rules-1.0'
REVIEW = Decimal('45')
PRIORITY = Decimal('60')

def number(data, field, upper=None):
    value = data.get(field)
    if value is None or (isinstance(value, str) and not value.strip()):
        raise ValueError(f'{field}: відсутнє значення')
    if isinstance(value, bool) or not isinstance(value, (int, float, str, Decimal)):
        raise ValueError(f'{field}: очікується число')
    try:
        value = Decimal(str(value))
    except InvalidOperation:
        raise ValueError(f'{field}: очікується число') from None
    if not value.is_finite() or value < 0 or (upper is not None and value > upper):
        raise ValueError(f'{field}: поза допустимим діапазоном')
    return value

def checkpoint(data):
    value = data.get('task_completed')
    if isinstance(value, bool):
        return value
    if isinstance(value, str) and value.strip().lower() in ('true', 'false'):
        return value.strip().lower() == 'true'
    raise ValueError('task_completed: очікується true або false')

def assess(data):
    """Return score, factors and version, or an explicit unavailable result."""
    try:
        score = number(data, 'test_score', 100)
        progress = number(data, 'course_completion', 100)
        minutes = number(data, 'study_time')
        complete = checkpoint(data)
    except ValueError as error:
        return dict(status='unavailable', risk_score=None, factors={},
                    version=VERSION, reason=str(error))

    raw = {
        'test_score': Decimal('0.40') * (100 - score),
        'course_completion': Decimal('0.35') * (100 - progress),
        'task_completed': Decimal(0 if complete else 15),
        'study_time': Decimal(10) * (1 - min(minutes, Decimal(200)) / 200),
    }
    # Round only the final score used for routing. Keep extra precision in factors.
    index = sum(raw.values()).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)
    return dict(status='ok', risk_score=float(index),
                factors={k: float(v) for k, v in raw.items()},
                version=VERSION, reason=None)

def route(result):
    """C7 maps the score to a queue; a teacher decides on any intervention."""
    if result['status'] != 'ok':
        return 'unavailable'
    score = Decimal(str(result['risk_score']))
    if score >= PRIORITY:
        return 'priority'
    if score >= REVIEW:
        return 'review'
    return 'monitor'


## Тестові записи і фактична перевірка
Очікувані маршрути задані заздалегідь. S04 і S05 перевіряють рівність порогам, S06 перевіряє допустимість нульової активності, S07 і S08 — відмову.

In [2]:
SAMPLES = 'case_id,test_score,course_completion,task_completed,study_time,expected_route\nS01,94,90,true,210,monitor\nS02,45,48,false,60,priority\nS03,50,60,false,80,review\nS04,50,40,true,120,review\nS05,50,50,false,50,priority\nS06,75,66,true,0,monitor\nS07,,56,false,80,unavailable\nS08,150,45,false,80,unavailable\n'

import csv
from io import StringIO

samples = list(csv.DictReader(StringIO(SAMPLES.strip())))
checks = []
for case in samples:
    result = assess(case)
    decision = route(result)
    ok = decision == case['expected_route']
    checks.append(ok)
    detail = result['reason'] or ''
    print(f"{case['case_id']}  {result['risk_score']!s:>5}  {decision:12}  "
          f"очікувано: {case['expected_route']:12}  {'так' if ok else 'НІ'}  {detail}")
print(f"\nЗбіг маршрутів: {sum(checks)}/{len(checks)} ({sum(checks)/len(checks):.0%})")
assert all(checks)
print("Складові сценарію S02:", assess(samples[1])['factors'])

S01    5.9  monitor       очікувано: monitor       так  
S02   62.2  priority      очікувано: priority      так  
S03   55.0  review        очікувано: review        так  
S04   45.0  review        очікувано: review        так  
S05   60.0  priority      очікувано: priority      так  
S06   31.9  monitor       очікувано: monitor       так  
S07   None  unavailable   очікувано: unavailable   так  test_score: відсутнє значення
S08   None  unavailable   очікувано: unavailable   так  test_score: поза допустимим діапазоном

Збіг маршрутів: 8/8 (100%)
Складові сценарію S02: {'test_score': 22.0, 'course_completion': 18.2, 'task_completed': 15.0, 'study_time': 7.0}


## Висновок
Збіг маршрутів на цих прикладах підтверджує працездатність логіки. Прогностичну якість потрібно оцінювати окремо.